# Assignment 06: Clinic Data Wrangling Pipeline (Midterm)

Follow `README.md`: it names every file to save in `output/`, with its first line and line count. Replace each **TODO**, then restart the kernel and **Run All** before you commit, so the saved outputs show your work.

In [1]:
# Supplied setup: run this cell first and leave it unchanged.
from pathlib import Path
import hashlib
import json
import subprocess
import sys

import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| pandas", pd.__version__)


def find_assignment_dir(start):
    """The folder holding data_manifest.json: this one, a parent, or 06/mid-term below one."""
    for folder in (start, *start.parents):
        for candidate in (folder, folder / "06" / "mid-term"):
            if (candidate / "data_manifest.json").is_file():
                return candidate
    raise FileNotFoundError("Open this notebook from the assignment folder that holds data_manifest.json.")


BASE_DIR = find_assignment_dir(Path.cwd().resolve())
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
PROFILE_PATH = OUTPUT_DIR / "table_profile.csv"
AUDIT_PATH = OUTPUT_DIR / "issue_audit.csv"
PATIENTS_SUMMARY_PATH = OUTPUT_DIR / "patients_summary.csv"
GENDER_CATEGORIES_PATH = OUTPUT_DIR / "gender_categories.csv"
VISITS_SUMMARY_PATH = OUTPUT_DIR / "visits_summary.csv"
DECISION_LOG_PATH = OUTPUT_DIR / "decision_log.csv"
A1C_SUMMARY_PATH = OUTPUT_DIR / "a1c_summary.csv"
MERGE_AUDIT_PATH = OUTPUT_DIR / "merge_audit.csv"
SBP_WIDE_SAMPLE_PATH = OUTPUT_DIR / "sbp_wide_sample.csv"
ANALYSIS_SAMPLE_PATH = OUTPUT_DIR / "analysis_sample.csv"

# The patients whose rows the two extracts hold (README Tasks 4.3 and 5.2).
SAMPLE_IDS = ["P00001", "P00004", "P00088", "P00260", "P00446", "P00487", "P00631", "P00851",
              "P02354", "P03836", "P05763", "P07510", "P14287", "P16486", "P17390", "P30001"]

# The data is generated, not committed: generate_data.py writes the same tables on every computer.
manifest = json.loads((BASE_DIR / "data_manifest.json").read_text(encoding="utf-8"))
if not all((DATA_DIR / name).is_file() for name in manifest["files"]):
    print(subprocess.run([sys.executable, "generate_data.py"], cwd=BASE_DIR, check=True,
                         capture_output=True, text=True).stdout.strip())


def read_raw(path):
    """A Parquet file with the dtypes it stores; a CSV file with every column as text, blanks kept as written."""
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype="string", keep_default_na=False)


raw = {Path(name).stem: read_raw(DATA_DIR / name) for name in manifest["files"]}
# The checksum covers each table's contents, written out as CSV text.
for name, expected in manifest["files"].items():
    text = raw[Path(name).stem].to_csv(index=False, lineterminator="\n")
    assert hashlib.sha256(text.encode("utf-8")).hexdigest() == expected["sha256"], (
        f"data/{name} differs from the generated file; delete the data folder and run this cell again.")
print(f"Verified {len(manifest['files'])} data files")
raw_snapshot = {name: table.copy(deep=True) for name, table in raw.items()}
list(raw)

Python 3.13.7 | NumPy 2.3.3 | pandas 3.0.5
Verified 5 data files


['patients', 'sites', 'visits_north', 'visits_south', 'a1c_wide']

## Cumulative midterm checkpoint (human review)

- **Lecture 01 (command line):** Task 1.1 uses `>` redirection in `wc -l data/*.csv > output/file_inventory.txt` to save the line count of each CSV file into `output/file_inventory.txt` instead of printing it.
- **Lecture 02 (functions):** Task 1.2 defines `profile_table(name, table)`, which returns a dictionary of rows, columns, and exact duplicates, and calls it once per file in a loop over `manifest["files"]` to build `output/table_profile.csv`.
- **Lecture 03 (NumPy):** After Task 4.3, `np.array(complete_pairs, dtype=float)` has shape (27149, 2), one row per patient with both readings and one column per visit, and `np.mean(sbp_array, axis=0)` gives 143.01 mmHg at baseline and 137.15 mmHg at week 12; I used `axis=0` because it collapses the patient rows and leaves one mean per visit column.
- **Lecture 04 (labeled pandas):** Task 4.1 selects rows by label with a boolean mask and `.loc`, as in `a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean()`, to compute each visit's mean A1c for `output/a1c_summary.csv`.
- **Lecture 05 (cleaning and validation):** Task 3.1 maps gender spellings to F, M, or X with a dictionary and `Series.map()`, and Task 5.1 collects named True/False checks in a Series and stops with `assert checks.all(), checks[~checks]` before `analysis_sample.csv` is saved.
- **Lecture 06 (joins, reshaping, concatenation):** Task 2.2 stacks the two visit files with `pd.concat(..., ignore_index=True)` after adding a `source_file` column, Task 4.3 reshapes blood pressure with `pivot()` after keeping the later recheck, and Task 4.4 checks each left merge's cardinality with `validate=`.

## Task 1: Inventory the files

### 1.1 Count lines from the terminal

Make `output/file_inventory.txt` in the terminal (README Task 1.1), then paste the command you ran into the `%%bash` cell below.

In [2]:
%%bash
wc -l data/*.csv > output/file_inventory.txt

### 1.2 Profile each table with a function

In [3]:
def profile_table(name, table):
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "exact_duplicate_rows": int(table.duplicated().sum()),
    }

results = []
for name in manifest["files"]:
    results.append(profile_table(name, raw[Path(name).stem]))

table_profile = pd.DataFrame(results)
table_profile.to_csv(PROFILE_PATH, index=False)
table_profile

,table,rows,columns,exact_duplicate_rows
0,patients.parquet,30045,4,45
1,sites.csv,5,3,0
2,visits_north.csv,26297,6,35
3,visits_south.csv,31074,6,35
4,a1c_wide.parquet,29424,3,0


## Task 2: Contract, stack, and audit

### 2.1 Write the data contract (human review)

| Input table | Cleaned table | One row is | Key |
|---|---|---|---|
| patients.parquet | patients_clean | one registered patient | `patient_id` (single) |
| sites.csv | sites_current | one current site record | `site_code` (single) |
| visits_north.csv | visits_clean (stacked with south) | one visit of one patient, keeping the later recheck | `patient_id` + `visit` (composite) |
| visits_south.csv | visits_clean (stacked with north) | one visit of one patient, keeping the later recheck | `patient_id` + `visit` (composite) |
| a1c_wide.parquet | a1c_long (melted) | one A1c result for one patient at one visit | `patient_id` + `visit` (composite) |

**Raw rows that violate these keys:**

- patients.parquet: 45 exact repeat rows duplicate an earlier `patient_id`.
- visits_north.csv and visits_south.csv: 35 exact repeat rows in each; rechecks, where the same `patient_id` and `visit` were measured twice at different `visit_time`s; and visit spellings such as `wk12` that only show up as repeats once they are mapped to `baseline` or `week12`.
- sites.csv: N01 and S02 each appear twice, once as a `current` record and once as a `retired` one (4 rows), so `site_code` is unique only after keeping current records.

**Expected join cardinality:**

- visits_clean → patients_clean on `patient_id`: many-to-one (a patient has up to two visits).
- visits_clean → sites_current on `site_code`: many-to-one (many patients and visits per site).
- visits_clean → a1c_long on `patient_id` + `visit`: one-to-one (at most one visit and one result per pair).

### 2.2 Stack the two visit files

In [4]:
north = raw["visits_north"].copy(deep=True)
south = raw["visits_south"].copy(deep=True).rename(columns={"wt_kg": "weight_kg"})

north["source_file"] = "visits_north.csv"
south["source_file"] = "visits_south.csv"

visits_raw = pd.concat([north, south], ignore_index=True)
print(visits_raw.shape)
print(visits_raw.isna().all())
visits_raw.head()

(57371, 7)
patient_id     False
visit          False
visit_date     False
visit_time     False
sbp_mmhg       False
weight_kg      False
source_file    False
dtype: bool


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
0,P00002,baseline,2026-02-20,09:50,132,85.2,visits_north.csv
1,P00002,week12,2026-05-08,15:20,129,82.0,visits_north.csv
2,P00003,baseline,2026-01-16,11:45,148,67.5,visits_north.csv
3,P00003,wk12,2026-04-07,10:00,137,64.0,visits_north.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv


In [5]:
GENDER_MAP = {
    "f": "F", "female": "F", "woman": "F",
    "m": "M", "male": "M", "man": "M",
    "x": "X", "nb": "X", "nonbinary": "X", "non-binary": "X", "non binary": "X",
}
VISIT_MAP = {
    "baseline": "baseline", "base": "baseline", "bl": "baseline", "visit 1": "baseline",
    "week12": "week12", "wk12": "week12", "week 12": "week12", "wk 12": "week12",
    "week-12": "week12", "week_12": "week12", "w12": "week12",
}

g = raw["patients"]["gender"].fillna("").str.strip().str.lower()
print("Unmapped gender spellings:")
print(g[~g.isin(GENDER_MAP.keys())].value_counts(dropna=False).to_string())

v = visits_raw["visit"].fillna("").str.strip().str.lower()
print("\nUnmapped visit spellings:")
print(v[~v.isin(VISIT_MAP.keys())].value_counts(dropna=False).to_string())

Unmapped gender spellings:
gender
           486
unknown    378

Unmapped visit spellings:
Series([], )


### 2.3 Audit the raw tables

In [6]:
p = raw["patients"]

# patients: gender
gender = p["gender"].fillna("")
gender_empty = gender.str.strip() == ""
gender_not_code = ~gender_empty & ~gender.isin(["F", "M", "X"])

# patients: birth_year sentinel
birth_9999 = pd.to_numeric(p["birth_year"].str.strip(), errors="coerce") == 9999

# patients: site codes that change under Task 3.1's rule
site = p["site_code"].fillna("")
site_norm = site.str.upper().str.replace("-", "").str.replace(" ", "")
site_needs_norm = site != site_norm

# sites: every row whose site_code appears on another row
sites_repeated = raw["sites"].duplicated(subset=["site_code"], keep=False)

# visits: labels not exactly baseline or week12
visit_not_label = ~visits_raw["visit"].isin(["baseline", "week12"])

# visits: sbp sentinel and out-of-range (999 counted only as sentinel)
sbp = pd.to_numeric(visits_raw["sbp_mmhg"].str.strip(), errors="coerce")
sbp_999 = sbp == 999
sbp_out = sbp.notna() & ~sbp_999 & ((sbp < 70) | (sbp > 250))

# visits: dates that are not exact YYYY-MM-DD calendar dates
d = visits_raw["visit_date"]
exact_text = d.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
real_date = pd.to_datetime(d, format="%Y-%m-%d", errors="coerce").notna()
date_fail = ~(exact_text & real_date)

# visits: rows in repeated patient_id + mapped visit pairs
pairs = pd.DataFrame({
    "patient_id": visits_raw["patient_id"],
    "visit": visits_raw["visit"].str.strip().str.lower().map(VISIT_MAP),
})
pair_repeat = pairs.duplicated(subset=["patient_id", "visit"], keep=False)

issue_audit = pd.DataFrame({
    "issue": [
        "patients: gender values that are not a code",
        "patients: empty gender",
        "patients: birth_year sentinel 9999",
        "patients: site codes needing normalization",
        "sites: rows with a repeated site_code",
        "visits: visit labels that are not baseline or week12",
        "visits: sbp sentinel 999",
        "visits: sbp outside 70 through 250",
        "visits: date parse failures",
        "visits: rows in repeated patient_id and visit pairs",
    ],
    "count": [int(x.sum()) for x in [
        gender_not_code, gender_empty, birth_9999, site_needs_norm, sites_repeated,
        visit_not_label, sbp_999, sbp_out, date_fail, pair_repeat,
    ]],
})
issue_audit.to_csv(AUDIT_PATH, index=False)

assert all(raw[name].equals(raw_snapshot[name]) for name in raw), "Auditing must not change raw."
issue_audit

,issue,count
0,patients: gender values that are not a code,6968
1,patients: empty gender,486
2,patients: birth_year sentinel 9999,371
3,patients: site codes needing normalization,1811
4,sites: rows with a repeated site_code,4
5,visits: visit labels that are not baseline or ...,2309
6,visits: sbp sentinel 999,341
7,visits: sbp outside 70 through 250,116
8,visits: date parse failures,169
9,visits: rows in repeated patient_id and visit ...,380


## Task 3: Clean

### 3.1 Clean the patients

In [7]:
display(raw["patients"]["gender"].value_counts(dropna=False))

patients_clean = raw["patients"].copy(deep=True)

# 1. exact duplicate rows
patients_clean = patients_clean.drop_duplicates(keep="first")

# 2. gender: strip, lowercase, map to F / M / X; unknown and blank become missing
patients_clean["gender"] = (
    patients_clean["gender"].str.strip().str.lower().map(GENDER_MAP).astype("category")
)

# 3. birth_year: 9999 -> missing, Int64
birth = pd.to_numeric(patients_clean["birth_year"].str.strip(), errors="coerce").astype("Int64")
patients_clean["birth_year"] = birth.replace(9999, pd.NA)

# 4. site_code: uppercase, no hyphens or spaces
patients_clean["site_code"] = (
    patients_clean["site_code"].str.upper().str.replace("-", "").str.replace(" ", "")
)

# 5. renumber
patients_clean = patients_clean.reset_index(drop=True)

print(patients_clean["gender"].cat.categories)
print(len(patients_clean))
patients_clean.dtypes

gender
F             11248
M             10971
male           1039
female         1018
Male            622
f               622
Female          578
m               538
woman           400
                382
X               372
man             329
unknown         282
MALE            256
FEMALE          231
 M              223
 F              205
Woman           131
nonbinary       118
                104
NB              104
Unknown          96
non-binary       80
x                62
Non-Binary       34
Name: count, dtype: int64[pyarrow]

Index(['F', 'M', 'X'], dtype='str')
30000


patient_id      string
gender        category
birth_year       Int64
site_code       string
dtype: object

In [8]:
print(patients_clean["gender"].cat.categories)

patients_summary = pd.DataFrame({
    "metric": [
        "patients", "birth_year missing", "birth_year min", "birth_year max",
        "site_code values", "patients at N01", "patients at S02", "patients at E04",
    ],
    "value": [
        len(patients_clean),
        int(patients_clean["birth_year"].isna().sum()),
        int(patients_clean["birth_year"].min()),
        int(patients_clean["birth_year"].max()),
        int(patients_clean["site_code"].nunique()),
        int((patients_clean["site_code"] == "N01").sum()),
        int((patients_clean["site_code"] == "S02").sum()),
        int((patients_clean["site_code"] == "E04").sum()),
    ],
})
patients_summary.to_csv(PATIENTS_SUMMARY_PATH, index=False)
display(patients_summary)

gender_counts = patients_clean["gender"].value_counts(dropna=False).reset_index()
gender_counts["code"] = gender_counts["gender"].cat.codes
gender_counts.to_csv(GENDER_CATEGORIES_PATH, index=False)
gender_counts

Index(['F', 'M', 'X'], dtype='str')


,metric,value
0,patients,30000
1,birth_year missing,371
2,birth_year min,1941
3,birth_year max,1996
4,site_code values,3
5,patients at N01,13739
6,patients at S02,15706
7,patients at E04,555


,gender,count,code
0,F,14413,0
1,M,13955,1
2,NaN,863,-1
3,X,769,2


### 3.2 Clean the visits

In [9]:
visits_clean = visits_raw.copy(deep=True)

# 1. exact duplicate rows
visits_clean = visits_clean.drop_duplicates(keep="first")

# 2. visit: list spellings, then map to baseline / week12
display(visits_clean["visit"].value_counts(dropna=False))
visits_clean["visit"] = visits_clean["visit"].str.strip().str.lower().map(VISIT_MAP)

# 3. sbp_mmhg: keep 70 through 250; everything else (999 included) -> missing; Int64
sbp = pd.to_numeric(visits_clean["sbp_mmhg"].str.strip(), errors="coerce")
in_range = sbp.between(70, 250).fillna(False).astype(bool)
visits_clean["sbp_mmhg"] = sbp.where(in_range).astype("Int64")

# 4. weight_kg: numeric; blanks stay missing
visits_clean["weight_kg"] = pd.to_numeric(visits_clean["weight_kg"].str.strip(), errors="coerce")

# 5. visit_date: only exact YYYY-MM-DD text for a real calendar date
d = visits_clean["visit_date"]
exact = d.str.fullmatch(r"\d{4}-\d{2}-\d{2}").fillna(False).astype(bool)
parsed = pd.to_datetime(d, format="%Y-%m-%d", errors="coerce")
visits_clean["visit_date"] = parsed.where(exact)

print(len(visits_clean))
print(visits_clean["visit"].value_counts(dropna=False))
visits_clean.dtypes

visit
baseline     28496
week12       26498
BASELINE       255
Baseline       242
bl             235
baseline       228
BL             220
wk12           204
 week12        198
Week 12        194
WEEK12         186
Wk12           177
week 12        168
Name: count, dtype: int64[pyarrow]

57301
visit
baseline    29676
week12      27625
Name: count, dtype: int64


patient_id             string
visit                     str
visit_date     datetime64[us]
visit_time             string
sbp_mmhg                Int64
weight_kg             Float64
source_file               str
dtype: object

In [10]:
# 6. show rechecks: same patient_id and visit appearing more than once
repeat = visits_clean.duplicated(subset=["patient_id", "visit"], keep=False)
print("rows in repeated pairs:", repeat.sum())
display(visits_clean[repeat].sort_values(["patient_id", "visit", "visit_time"]).head(10))

# keep the later reading of each pair
visits_clean = (
    visits_clean.sort_values("visit_time", kind="stable")
    .drop_duplicates(subset=["patient_id", "visit"], keep="last")
)
print("rows after keeping later reading:", len(visits_clean))
print("pairs still repeated:", visits_clean.duplicated(subset=["patient_id", "visit"]).sum())

rows in repeated pairs: 240


,patient_id,visit,visit_date,visit_time,sbp_mmhg,weight_kg,source_file
26298,P00001,baseline,2026-03-06,14:15,141,52.4,visits_south.csv
26297,P00001,baseline,2026-03-06,14:40,146,52.4,visits_south.csv
4,P00004,baseline,2026-02-25,10:55,145,106.2,visits_north.csv
5,P00004,baseline,2026-02-25,11:15,142,106.2,visits_north.csv
84,P00088,baseline,2026-03-10,09:30,153,68.2,visits_north.csv
85,P00088,baseline,2026-03-10,09:50,152,68.2,visits_north.csv
26491,P00183,week12,2026-05-25,14:15,134,78.9,visits_south.csv
26490,P00183,week12,2026-05-25,14:35,133,78.9,visits_south.csv
26684,P00377,baseline,2026-01-30,09:05,147,101.0,visits_south.csv
26685,P00377,baseline,2026-01-30,09:15,139,101.0,visits_south.csv


rows after keeping later reading: 57181
pairs still repeated: 0


In [11]:
# 7. sort, renumber, keep the six columns
visits_clean = visits_clean.sort_values(["patient_id", "visit"]).reset_index(drop=True)
visits_clean = visits_clean[["patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "source_file"]]
display(visits_clean["visit"].value_counts(dropna=False))

visits_summary = pd.DataFrame({
    "metric": [
        "visits", "baseline visits", "week12 visits",
        "visit_date missing", "visit_date min", "visit_date max",
        "sbp_mmhg missing", "sbp_mmhg min", "sbp_mmhg max",
        "weight_kg missing", "weight_kg mean",
        "visits from visits_north.csv", "visits from visits_south.csv",
    ],
    "value": [
        len(visits_clean),
        int((visits_clean["visit"] == "baseline").sum()),
        int((visits_clean["visit"] == "week12").sum()),
        int(visits_clean["visit_date"].isna().sum()),
        str(visits_clean["visit_date"].min().date()),
        str(visits_clean["visit_date"].max().date()),
        int(visits_clean["sbp_mmhg"].isna().sum()),
        int(visits_clean["sbp_mmhg"].min()),
        int(visits_clean["sbp_mmhg"].max()),
        int(visits_clean["weight_kg"].isna().sum()),
        round(float(visits_clean["weight_kg"].mean()), 2),
        int((visits_clean["source_file"] == "visits_north.csv").sum()),
        int((visits_clean["source_file"] == "visits_south.csv").sum()),
    ],
})
visits_summary.to_csv(VISITS_SUMMARY_PATH, index=False)
visits_summary

visit
baseline    29599
week12      27582
Name: count, dtype: int64

,metric,value
0,visits,57181
1,baseline visits,29599
2,week12 visits,27582
3,visit_date missing,169
4,visit_date min,2026-01-05
5,visit_date max,2026-07-06
6,sbp_mmhg missing,456
7,sbp_mmhg min,70
8,sbp_mmhg max,250
9,weight_kg missing,828


### 3.3 Record the decisions

In [12]:
decision_table = pd.DataFrame(
    [
        ["patients", "exact duplicate rows",
         "Dropped the 45 exact repeat rows with drop_duplicates(keep='first'), leaving 30000 patients.",
         "An identical row is the same registration entered twice, so keeping one copy makes patient_id unique without losing information."],
        ["patients", "gender spellings that are not a code",
         "Stripped and lowercased gender, mapped female/woman/f to F, male/man/m to M, and nonbinary/non-binary/nb/x to X with Series.map(), set unknown and blanks to missing, and stored it as a category.",
         "The 6968 non-code spellings and 486 blanks all carry one of three meanings or none, and the data dictionary allows only F, M, X, or missing."],
        ["patients", "site codes needing normalization",
         "Uppercased site_code and removed every hyphen and space, so the 1811 variants such as n-01 become N01, S02, or E04.",
         "The variants are typing differences for the same three registry sites, and normalized codes are needed to join patients to sites."],
        ["patients", "birth_year sentinel 9999",
         "Converted birth_year to Int64 and replaced the 371 values of 9999 with missing.",
         "9999 means not recorded, so leaving it as a number would distort ages and the birth_year min and max."],
        ["visits", "weight column named wt_kg in visits_south.csv",
         "Renamed wt_kg to weight_kg in a copy of visits_south before stacking it under visits_north.",
         "Both files record the same measurement, so one column name lets the 31074 south rows stack into weight_kg instead of a mostly empty extra column."],
        ["visits", "sbp sentinel 999 and readings outside 70 through 250",
         "Converted sbp_mmhg to numbers, kept 70 through 250 inclusive as Int64, and set the 341 values of 999 and the 116 out-of-range values to missing.",
         "999 means not measured and readings outside 70 to 250 mmHg are not plausible clinic values, while 70 and 250 themselves are valid boundary readings."],
        ["visits", "impossible and loosely written visit dates",
         "Kept visit_date only when the text is exact YYYY-MM-DD and parses to a real calendar date, setting the 169 others to missing.",
         "Loose formats are ambiguous and impossible dates such as 2026-02-30 cannot be real visits, so guessing would plant wrong dates."],
        ["visits", "repeated patient_id and visit pairs",
         "After removing exact repeats, showed the 120 pairs measured twice and kept the later reading by visit_time with drop_duplicates(keep='last').",
         "A same-day recheck replaces the first reading, and keeping one row per patient and visit is what the pivot and the joins require."],
        ["sites", "repeated site_code rows",
         "Kept only rows whose record_status is current, so N01 and S02 each keep one name and the retired versions are dropped.",
         "N01 and S02 each have a current and a retired record, and only one row per site_code lets the visit-to-site join stay many-to-one."],
    ],
    columns=["table", "issue", "decision", "reason"],
)
decision_table.to_csv(DECISION_LOG_PATH, index=False)
decision_table

,table,issue,decision,reason
0,patients,exact duplicate rows,Dropped the 45 exact repeat rows with drop_dup...,An identical row is the same registration ente...
1,patients,gender spellings that are not a code,"Stripped and lowercased gender, mapped female/...",The 6968 non-code spellings and 486 blanks all...
2,patients,site codes needing normalization,Uppercased site_code and removed every hyphen ...,The variants are typing differences for the sa...
3,patients,birth_year sentinel 9999,Converted birth_year to Int64 and replaced the...,"9999 means not recorded, so leaving it as a nu..."
4,visits,weight column named wt_kg in visits_south.csv,Renamed wt_kg to weight_kg in a copy of visits...,"Both files record the same measurement, so one..."
5,visits,sbp sentinel 999 and readings outside 70 throu...,"Converted sbp_mmhg to numbers, kept 70 through...",999 means not measured and readings outside 70...
6,visits,impossible and loosely written visit dates,Kept visit_date only when the text is exact YY...,Loose formats are ambiguous and impossible dat...
7,visits,repeated patient_id and visit pairs,"After removing exact repeats, showed the 120 p...","A same-day recheck replaces the first reading,..."
8,sites,repeated site_code rows,"Kept only rows whose record_status is current,...",N01 and S02 each have a current and a retired ...


## Task 4: Reshape and join

### 4.1 Melt the A1c results to long form

In [13]:
a1c_long = raw["a1c_wide"].melt(
    id_vars=["patient_id"],
    value_vars=["a1c_baseline", "a1c_week12"],
    var_name="visit",
    value_name="a1c_pct",
)
a1c_long["visit"] = a1c_long["visit"].map({"a1c_baseline": "baseline", "a1c_week12": "week12"})

print(a1c_long["a1c_pct"].dtype)   # expect float64
a1c_long = a1c_long.dropna(subset=["a1c_pct"]).reset_index(drop=True)

a1c_summary = pd.DataFrame({
    "visit": ["baseline", "week12"],
    "results": [
        int((a1c_long["visit"] == "baseline").sum()),
        int((a1c_long["visit"] == "week12").sum()),
    ],
    "a1c_pct_mean": [
        round(float(a1c_long.loc[a1c_long["visit"] == "baseline", "a1c_pct"].mean()), 2),
        round(float(a1c_long.loc[a1c_long["visit"] == "week12", "a1c_pct"].mean()), 2),
    ],
})
a1c_summary.to_csv(A1C_SUMMARY_PATH, index=False)
print(len(a1c_long))
a1c_summary

float64
56760


,visit,results,a1c_pct_mean
0,baseline,29107,8.15
1,week12,27653,7.63


### 4.2 Check the keys before joining

In [14]:
sites = raw["sites"]

# 1. merging on the raw site list breaks many_to_one
try:
    patients_clean.merge(sites, on="site_code", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)

# 2. show the repeated site rows, then keep current records
display(sites[sites.duplicated(subset=["site_code"], keep=False)])
sites_current = sites.loc[sites["record_status"] == "current", ["site_code", "site_name"]].reset_index(drop=True)
sites_current

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 site_code
      N01
      S02 ...


,site_code,site_name,record_status
0,N01,North Clinic,current
1,N01,North Street Clinic,retired
2,S02,Southgate Clinic,retired
3,S02,South Valley Clinic,current


,site_code,site_name
0,N01,North Clinic
1,S02,South Valley Clinic
2,W03,West Clinic


In [15]:
# 3. outer merge of visits and A1c results
visit_a1c = visits_clean.merge(a1c_long, on=["patient_id", "visit"], how="outer", indicator=True)
merge_counts = visit_a1c["_merge"].value_counts()
display(merge_counts)

merge_audit = pd.DataFrame({
    "check": [
        "visits with an A1c result",
        "visits without an A1c result",
        "A1c results without a visit",
        "visits whose patient is not in the registry",
        "registry patients with no visit",
        "registry patients whose site is not in the site list",
        "site list rows removed by the current-record rule",
    ],
    "count": [
        int(merge_counts["both"]),
        int(merge_counts["left_only"]),
        int(merge_counts["right_only"]),
        int((~visits_clean["patient_id"].isin(patients_clean["patient_id"])).sum()),
        int((~patients_clean["patient_id"].isin(visits_clean["patient_id"])).sum()),
        int((~patients_clean["site_code"].isin(sites["site_code"])).sum()),
        len(sites) - len(sites_current),
    ],
})
merge_audit.to_csv(MERGE_AUDIT_PATH, index=False)
merge_audit

_merge
both          54006
left_only      3175
right_only     2754
Name: count, dtype: int64

,check,count
0,visits with an A1c result,54006
1,visits without an A1c result,3175
2,A1c results without a visit,2754
3,visits whose patient is not in the registry,79
4,registry patients with no visit,441
5,registry patients whose site is not in the sit...,555
6,site list rows removed by the current-record rule,2


### 4.3 Pivot blood pressure to one row per patient

In [16]:
sbp_wide = visits_clean.pivot(index="patient_id", columns="visit", values="sbp_mmhg").reset_index()
sbp_wide.columns.name = None
sbp_wide["sbp_change"] = sbp_wide["week12"] - sbp_wide["baseline"]

sbp_wide_sample = sbp_wide[sbp_wide["patient_id"].isin(SAMPLE_IDS)]
sbp_wide_sample.to_csv(SBP_WIDE_SAMPLE_PATH, index=False)
print(sbp_wide.shape, sbp_wide["patient_id"].is_unique)
sbp_wide_sample

(29599, 4) True


,patient_id,baseline,week12,sbp_change
0,P00001,146,129,-17
3,P00004,142,<NA>,<NA>
87,P00088,152,147,-5
259,P00260,70,138,68
445,P00446,250,144,-106
486,P00487,<NA>,143,<NA>
849,P00851,<NA>,129,<NA>
2336,P02354,127,125,-2
3789,P03836,134,125,-9
5684,P05763,161,151,-10


#### NumPy checkpoint (Lecture 03, human review)

From the full `sbp_wide`, select `baseline` and `week12` in that order, keeping only patients with both readings. Name this two-column table `complete_pairs`; `np.array(complete_pairs, dtype=float)` converts it to a NumPy array. Display its shape and calculate one NumPy mean per visit across patients. Save the outputs and explain your axis choice in the cumulative checkpoint's Lecture 03 bullet. No extra output file.

In [17]:
complete_pairs = sbp_wide[["baseline", "week12"]].dropna()
sbp_array = np.array(complete_pairs, dtype=float)
display(sbp_array.shape)
np.mean(sbp_array, axis=0)

(27149, 2)

array([143.01337066, 137.15061328])

### 4.4 Join the analysis table

In [18]:
analysis = visits_clean.merge(patients_clean, on="patient_id", how="left", validate="many_to_one")
print("after patients:", len(analysis))
analysis = analysis.merge(sites_current, on="site_code", how="left", validate="many_to_one")
print("after sites:", len(analysis))
analysis = analysis.merge(a1c_long, on=["patient_id", "visit"], how="left", validate="one_to_one")
print("after A1c:", len(analysis))

analysis = analysis[[
    "patient_id", "visit", "visit_date", "sbp_mmhg", "weight_kg", "a1c_pct",
    "gender", "birth_year", "site_code", "site_name", "source_file",
]]
analysis = analysis.astype({
    "patient_id": "string",
    "visit": "string",
    "visit_date": "datetime64[us]",
    "sbp_mmhg": "Int64",
    "weight_kg": "float64",
    "a1c_pct": "float64",
    "gender": "string",
    "birth_year": "Int64",
    "site_code": "string",
    "site_name": "string",
    "source_file": "string",
})
display(analysis.dtypes)
analysis.head()

after patients: 57181
after sites: 57181
after A1c: 57181


patient_id             string
visit                  string
visit_date     datetime64[us]
sbp_mmhg                Int64
weight_kg             float64
a1c_pct               float64
gender                 string
birth_year              Int64
site_code              string
site_name              string
source_file            string
dtype: object

,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00002,baseline,2026-02-20,132,85.2,8.4,F,<NA>,N01,North Clinic,visits_north.csv
3,P00002,week12,2026-05-08,129,82.0,7.4,F,<NA>,N01,North Clinic,visits_north.csv
4,P00003,baseline,2026-01-16,148,67.5,5.9,F,1970,N01,North Clinic,visits_north.csv


## Task 5: Validate, save, and read back

### 5.1 Validate before saving

In [19]:
current_names = sites_current["site_name"]

checks = pd.Series({
    "analysis rows equal visits_clean rows": len(analysis) == len(visits_clean),
    "no repeated patient_id and visit pair": not analysis.duplicated(subset=["patient_id", "visit"]).any(),
    "every present sbp_mmhg is 70 through 250": bool(analysis["sbp_mmhg"].dropna().between(70, 250).all()),
    "every present site_name is a current site name": bool(analysis["site_name"].dropna().isin(current_names).all()),
    "sbp_wide patient_id is unique": bool(sbp_wide["patient_id"].is_unique),
    "every raw table equals its snapshot": all(raw[name].equals(raw_snapshot[name]) for name in raw),
})
display(checks)

assert checks.all(), checks[~checks]

analysis rows equal visits_clean rows             True
no repeated patient_id and visit pair             True
every present sbp_mmhg is 70 through 250          True
every present site_name is a current site name    True
sbp_wide patient_id is unique                     True
every raw table equals its snapshot               True
dtype: bool

### 5.2 Save the analysis extract

In [20]:
analysis_sample = analysis[analysis["patient_id"].isin(SAMPLE_IDS)].reset_index(drop=True)
analysis_sample.to_csv(ANALYSIS_SAMPLE_PATH, index=False)
print(len(analysis_sample), visits_clean["patient_id"].isin(SAMPLE_IDS).sum())
analysis_sample

28 28


,patient_id,visit,visit_date,sbp_mmhg,weight_kg,a1c_pct,gender,birth_year,site_code,site_name,source_file
0,P00001,baseline,2026-03-06,146,52.4,7.3,F,1966,S02,South Valley Clinic,visits_south.csv
1,P00001,week12,2026-05-27,129,50.2,NaN,F,1966,S02,South Valley Clinic,visits_south.csv
2,P00004,baseline,2026-02-25,142,106.2,7.3,X,<NA>,N01,North Clinic,visits_north.csv
3,P00088,baseline,2026-03-10,152,68.2,8.4,F,1980,N01,North Clinic,visits_north.csv
4,P00088,week12,2026-06-03,147,66.3,7.6,F,1980,N01,North Clinic,visits_north.csv
5,P00260,baseline,NaT,70,80.4,9.5,F,1979,E04,<NA>,visits_south.csv
6,P00260,week12,2026-06-10,138,78.6,8.0,F,1979,E04,<NA>,visits_south.csv
7,P00446,baseline,NaT,250,NaN,6.5,F,1946,S02,South Valley Clinic,visits_south.csv
8,P00446,week12,2026-04-02,144,94.3,6.8,F,1946,S02,South Valley Clinic,visits_south.csv
9,P00487,baseline,2026-03-02,<NA>,103.0,7.9,M,1959,S02,South Valley Clinic,visits_south.csv


### 5.3 Read the file back

In [21]:
round_trip = pd.read_csv(
    ANALYSIS_SAMPLE_PATH,
    dtype={
        "patient_id": "string",
        "visit": "string",
        "sbp_mmhg": "Int64",
        "weight_kg": "float64",
        "a1c_pct": "float64",
        "gender": "string",
        "birth_year": "Int64",
        "site_code": "string",
        "site_name": "string",
        "source_file": "string",
    },
    parse_dates=["visit_date"],
)
print(round_trip.equals(analysis_sample))

True
